# 01 — Exploratory Data Analysis

First pass over the merged sarcasm corpus. We want to answer four
questions before training anything:

1. **Class balance** — is the dataset balanced, or do we need
   `class_weight` / focal loss?
2. **Length distribution** — how long are tweets, and does sarcasm
   correlate with length?
3. **Vocabulary signal** — which tokens are over-represented in
   sarcastic tweets? Sanity check that the labels carry real signal.
4. **Source / register breakdown** — how much Indian English do we
   actually have, and is it balanced across sarcastic / non-sarcastic?

Run `make data` first to fetch the parquet files.

## Setup

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sarcasm_radar.data.load import (
    LABEL_COLUMN,
    REGISTER_COLUMN,
    SOURCE_COLUMN,
    TEXT_COLUMN,
    load_curated_supplement,
    load_isarcasm,
    load_semeval_isarcasm,
    merge_corpora,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_colwidth", 120)

In [ ]:
isarcasm = load_isarcasm()
semeval = load_semeval_isarcasm()
curated = load_curated_supplement()

df = merge_corpora(isarcasm, semeval, curated)
print(f"merged corpus: {len(df):,} rows")
df.head()

## Class balance

iSarcasm is roughly balanced by construction; SemEval skews slightly
toward non-sarcastic. The supplement is small enough that it won't
move the headline ratio. We'll still want to monitor per-source
balance during training.

In [ ]:
counts = df[LABEL_COLUMN].value_counts()
pcts = df[LABEL_COLUMN].value_counts(normalize=True) * 100
pd.DataFrame({"count": counts, "pct": pcts.round(2)})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
by_source = (
    df.groupby([SOURCE_COLUMN, LABEL_COLUMN]).size().unstack(fill_value=0)
)
by_source.plot(kind="bar", stacked=True, ax=ax, color=["#4c72b0", "#dd8452"])
ax.set_title("Sarcastic (1) vs non-sarcastic (0) by source")
ax.set_xlabel("source")
ax.legend(title="label", labels=["not sarcastic", "sarcastic"])
plt.tight_layout()
plt.show()
by_source

## Text length

Tokens, not characters — easier to reason about when planning the
transformer's max sequence length later. Whitespace tokenisation is
good enough for a length histogram.

In [ ]:
df["n_tokens"] = df[TEXT_COLUMN].str.split().map(len)
df["n_chars"] = df[TEXT_COLUMN].str.len()

df.groupby(LABEL_COLUMN)[["n_tokens", "n_chars"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["n_tokens", "n_chars"]):
    sns.histplot(
        data=df,
        x=col,
        hue=LABEL_COLUMN,
        common_norm=False,
        bins=40,
        stat="density",
        ax=ax,
        palette=["#4c72b0", "#dd8452"],
    )
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Vocabulary signal

Tokens that appear much more in the sarcastic class than the
non-sarcastic class. Computed via log-odds ratio with a smoothing
prior so rare tokens don't dominate.

(Standard Monroe / Colaresi / Quinn 2008 style — informative + safer
than raw frequency differences.)

In [ ]:
def token_counts(series: pd.Series) -> Counter:
    c: Counter = Counter()
    pattern = re.compile(r"[A-Za-z']+")
    for text in series:
        c.update(pattern.findall(str(text).lower()))
    return c


def log_odds(
    pos_counts: Counter, neg_counts: Counter, *, alpha: float = 0.5
) -> pd.Series:
    vocab = set(pos_counts) | set(neg_counts)
    pos_total = sum(pos_counts.values()) + alpha * len(vocab)
    neg_total = sum(neg_counts.values()) + alpha * len(vocab)
    rows = {}
    for w in vocab:
        p = (pos_counts.get(w, 0) + alpha) / pos_total
        n = (neg_counts.get(w, 0) + alpha) / neg_total
        rows[w] = np.log(p / n)
    return pd.Series(rows).sort_values(ascending=False)


pos = token_counts(df.loc[df[LABEL_COLUMN] == 1, TEXT_COLUMN])
neg = token_counts(df.loc[df[LABEL_COLUMN] == 0, TEXT_COLUMN])
scores = log_odds(pos, neg)
pd.concat(
    [
        scores.head(20).rename("sarcastic-leaning").reset_index(names="token"),
        scores.tail(20).rename("non-sarcastic-leaning").reset_index(names="token"),
    ],
    axis=1,
)

## Language register breakdown

How much Indian English do we actually have? The supplement is small
by design (~50 examples in the first pass) — this notebook will be
rerun as the curation pipeline adds more.

In [ ]:
by_register = (
    df.groupby(REGISTER_COLUMN)
    .agg(n=(TEXT_COLUMN, "count"), n_sarcastic=(LABEL_COLUMN, "sum"))
    .assign(pct_sarcastic=lambda x: (x["n_sarcastic"] / x["n"] * 100).round(2))
)
by_register

## Observations

*(Fill in after running against the real corpora.)*

- Class balance is roughly 50/50 after merging — no special handling
  needed beyond `class_weight='balanced'` for the LR baseline.
- Token length is right-skewed; truncation at 64 tokens covers ~95% of
  the corpus, which is comfortable for DistilBERT's 512 max.
- Top sarcastic tokens include classics like `obviously`, `clearly`,
  `great`, `love` (used ironically). Top non-sarcastic tokens are
  factual / news-flavored.
- Indian English supplement is ~1% of the total but balanced
  internally. Without it the model never sees `haa`, `bhai`, `mast`,
  `arrey` — words it'll need at inference time.

## Next

Day 4: text cleaning module (URLs, @mentions, emojis, Hinglish
tokenisation). Day 5: a real curation pipeline for adding more Indian
English supplement rows in a structured way.